In [ ]:
################################################## RUN THIS NOTEBOOK AND MENTAL ROBERTA ON GOOGLE COLAB ##################################################  

In [ ]:
import pandas as pd

df_distilbert = pd.read_csv("../data/cleaned/balanced_30k_dataset.csv")
df_distilbert.head()

,title,label
0,Fuck Hotlines Hot lines are for poor people wh...,1.0
1,"I don't want help, I want an end. I don't want...",1.0
2,Even after I die I feel like remains deserve t...,1.0
3,Gratitude 5 Brutal But Powerful Life Truths”,0.0
4,I am autistic and have aspergers. Meaning i ca...,1.0


In [4]:
from sklearn.model_selection import train_test_split

X = df_distilbert["title"]
y = df_distilbert["label"]

X_train_d, X_test_val_d, y_train_d, y_test_val_d = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_test_d, X_val_d, y_test_d, y_val_d = train_test_split(X_test_val_d, y_test_val_d, test_size=0.50, random_state=42, stratify=y_test_val_d)

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/home/thomas_d/.pyenv/versions/Certification/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from torch.utils.data import Dataset
from torch import tensor


class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        result = self.tokenizer(self.texts[idx], max_length=512, truncation=True, padding="max_length", return_tensors="pt")

        return {"input_ids": result["input_ids"].squeeze(0), "attention_mask": result["attention_mask"].squeeze(0), "label": tensor(self.labels[idx])}

In [ ]:
train_dataset = DepressionDataset(X_train_d.tolist(), y_train_d.tolist(), tokenizer)
test_dataset = DepressionDataset(X_test_d.tolist(), y_test_d.tolist(), tokenizer)
val_dataset = DepressionDataset(X_val_d.tolist(), y_val_d.tolist(), tokenizer)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
model.to(device)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
######## WARINING USE COLAB FOR TRAINING, THIS CODE IS NOT OPTIMIZED FOR CPU TRAINING ########
model.train()
for epoch in range(3):
    total_loss = 0
    batch_num = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        labels = labels.long()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch + 1}, Batch {batch_num + 1}, Loss: {loss.item():.4f}")
        batch_num += 1

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")

In [ ]:
model.eval()
all_preds = []
all_labels = []
batch_num = 0
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        batch_num += 1
        print(f"Validation Batch {batch_num} processed.")
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds))

In [ ]:
model.eval()
all_preds = []
all_labels = []
batch_num = 0
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        batch_num += 1
        print(f"Validation Batch {batch_num} processed.")
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds))